# 年轮 growth_ring · 电影运镜宣传视频（第三款）

渲染 `GrowthRingCinematic`（37 秒电影运镜版：年轮深潜 → 文字飞向镜头 → 悬浮画像 → 书页翻过 → 拉远收尾）。

这款与之前最大的区别：画面是一个 **3D 空间**，有虚拟摄像机持续推进/摇移/滚转，文字分布在不同深度，近大远小、前景遮挡、视差景深——是"拍出来的视频"而非"幻灯片 + 淡入"。

三款共用同一套代码和字体：
- 第一款（48 秒冲突钩子版）→ `colab_render.ipynb`
- 第二款（30 秒强动效版）→ `colab_render_kinetic.ipynb`
- 第三款（37 秒电影运镜版）→ 本文件

**流程**：① 环境 → ② 拉代码 → ③ 装依赖（带缓存）→ ④ 全片低分辨率预览 → ⑤ 正式渲染 → ⑥ 下载。

> 免费 Colab 用 CPU 渲染 1080×1920 / 37 秒，通常 10～25 分钟。建议先跑④全片预览，提前发现任意一帧的渲染错误。
>
> **改文案**：第三款文字集中在 `src/cinematic/content.ts`，改完 push 再重跑②。
> **调节奏**：五个镜头的秒数在 `src/cinematic/timeline.ts`。

In [ ]:
# ① 检查环境：Node 版本（Remotion 需要 Node >= 18）
import subprocess, re, os

def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"命令失败（退出码 {r.returncode}）: {cmd}")
    return r

node_ver = subprocess.run("node -v", shell=True, capture_output=True, text=True).stdout.strip()
print("当前 Node:", node_ver or "未安装")
m = re.search(r"v(\d+)", node_ver)
major = int(m.group(1)) if m else 0
if major < 18:
    print("Node 版本过低，安装 Node 22 ...")
    run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash -")
    run("apt-get install -y nodejs")
else:
    print("Node 版本满足要求。")
run("node -v && npm -v")

CPU_COUNT = min(os.cpu_count() or 2, 6)
print(f"检测到 {os.cpu_count()} 核，渲染并发设为 {CPU_COUNT}")

In [ ]:
# ② 拉取代码
# 公开仓库直接用地址；私有仓库换成：
#   https://<用户名>:<PersonalAccessToken>@github.com/<用户名>/<仓库名>.git
import os, shutil

REPO_URL = "https://github.com/frank-2312/growth_ring_promo.git"
BRANCH   = "main"
REPO_DIR = "/content/growth_ring_promo"
PROJ     = REPO_DIR  # 新仓库根目录就是 Remotion 项目本身

if os.path.isdir(f"{REPO_DIR}/.git"):
    print("仓库已存在，拉取最新代码 ...")
    run(f"cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}")
else:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    run(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}")

assert os.path.isdir(f"{PROJ}/src/kinetic"), "没找到 src/kinetic —— 代码不是最新版，重跑本格"
os.chdir(PROJ)
print("工作目录:", os.getcwd())
print("字体文件:", os.path.exists(f"{PROJ}/public/fonts/LXGWWenKai-Regular.ttf"))

### ③ 挂载 Drive 并缓存依赖

第一次把 `node_modules` 打包传到 Drive，以后重跑直接解压。若改了 `package.json`，把 `USE_CACHE` 设 `False` 重装一次。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

USE_CACHE = True
CACHE_ZIP = "/content/drive/MyDrive/growth_ring_node_modules.tar.gz"
NODE_MODULES = f"{PROJ}/node_modules"

os.chdir(PROJ)
if USE_CACHE and os.path.exists(CACHE_ZIP):
    print("从 Drive 恢复 node_modules ...")
    if os.path.exists(NODE_MODULES):
        shutil.rmtree(NODE_MODULES)
    run(f"tar -xzf {CACHE_ZIP} -C {PROJ}")
    print("已恢复。若依赖有变动，请把 USE_CACHE 改成 False 重跑。")
else:
    print("安装依赖并缓存到 Drive ...")
    run("npm install --no-audit --no-fund")
    run(f"tar -czf {CACHE_ZIP} -C {PROJ} node_modules")
    print("已缓存到 Drive，下次运行可跳过安装。")

### ④ 全片低分辨率预览（推荐先跑）

整段 37 秒扫一遍，分辨率压到一半。任何一帧的运行时错误都会在便宜的预览里暴露，而不是等正式渲染十几分钟后才报错。通过后再跑⑤。

想只看某段，临时加 `--frames=0-120`。

In [ ]:
import time
os.chdir(PROJ)
t0 = time.time()
run(f"cd {PROJ} && npx remotion render src/index.ts GrowthRingCinematic out/cinematic-preview.mp4 "
    f"--codec=h264 --scale=0.5 --concurrency={CPU_COUNT} --log=error")
print(f"预览渲染完成，用时 {time.time()-t0:.0f} 秒")

from google.colab import files
files.download(f"{PROJ}/out/cinematic-preview.mp4")

### ⑤ 正式渲染

全片 1080×1920 / 37 秒。免费 Colab 上请耐心等待，运行时不要断开。

In [ ]:
os.chdir(PROJ)
t0 = time.time()
run(f"cd {PROJ} && npx remotion render src/index.ts GrowthRingCinematic out/growth-ring-cinematic.mp4 "
    f"--codec=h264 --concurrency={CPU_COUNT} --log=error")
print(f"正式渲染完成，用时 {time.time()-t0:.0f} 秒")

In [ ]:
# ⑥ 下载成片
out = f"{PROJ}/out/growth-ring-cinematic.mp4"
if os.path.exists(out):
    print(f"文件大小: {os.path.getsize(out)/1024/1024:.1f} MB")
    files.download(out)
else:
    print("没有找到成片 —— 先跑⑤正式渲染。")

---

### 常见问题

- **npm install 卡住**：重跑③，并把 `USE_CACHE` 设 `False`。
- **Chrome 下载失败**：首次渲染会下载 Chrome Headless Shell（约 150MB）；受限网络下先跑 `npx remotion browser ensure`。
- **字体变方块**：确认②里字体文件存在；`public/fonts/` 的 ttf 随仓库同步，本地改了忘 push 就会出现。
- **改了文案没生效**：第三款文案在 `src/cinematic/content.ts`；改完要 push，再重跑②。
- **想改镜头运动**：镜头路径在 `src/cinematic/CinematicScenes.tsx` 每个场景的 `camPath(...)` 里，`scale` 是推进、`x/y` 是摇移、`roll` 是滚转。
- **想改节奏**：五个镜头的秒数在 `src/cinematic/timeline.ts`，改那里即可。
- **缓存依赖版本不对**：改了 `package.json` 后，把 `USE_CACHE=False` 跑一次③重建缓存。
- **渲染太慢**：加 `--scale=0.75` 降分辨率，或把并发调小。
- **所有 @remotion/* 版本必须一致**：升级时不要只升一个包。